# Four scientific workflows with Hallmark

EHT and DESI observations, Roman–Rubin simulations, and local DES Y3 products. The matching [CLI guide](scientific_workflows_cli.md) uses the same selected files; this notebook also shows one worktree.

Run **Setup**, then one numbered workflow from start to finish. Each uses a separate directory and one explicitly selected input. Choose a new `ROOT` to repeat an example. `repo.add(url)` catalogs remote files without downloading them. `Repo.clone(...)` copies an existing catalog; it preserves the complete catalog and history. Use `plan_download(include=...)` and approve transfers separately.

A URL template such as `.../uvfits/SR1_M87_2017_{day}_lo_hops_netcal_StokesI.uvfits` catalogs every matching file and records its fields as columns. These examples name one file exactly, without fields, so each catalogs a single input with its size and any published checksum.

Every transfer shows a plan and asks for approval. Declining or a failed transfer stops that workflow; do not continue to its versioning cell. The completed catalog remains available. The SSH example requires an existing export and your own access configuration.

The examples version a small `analysis-settings.txt` file as a local template beside each unchanged input. These settings record intended analysis choices; the cells do not process the scientific data. Detailed processing is described in prose.

## Setup

Install this checkout with `python -m pip install -e .` and install JupyterLab in the same environment with `python -m pip install jupyterlab`. Select that environment's kernel. Git needs your normal author configuration. No additional scientific-processing packages are needed for these examples.

Import Hallmark and the standard-library tools, then choose a new workspace under your home directory. The same setup works when Jupyter starts in the repository root or in `demo/`. Set `ROOT` before beginning a workflow. The source settings below select narrow dataset roots; replace `VM_URL` and `DES_LOCAL` with your existing export and local input when using those workflows.

In [ ]:
from pathlib import Path
import shutil
from hallmark import Repo

ROOT = (Path.home() / "hallmark-science-python").resolve()
ROOT.mkdir(exist_ok=True)
print(ROOT)

EHT_URL = "https://data.cyverse.org/dav-anon/iplant/commons/cyverse_curated/EHTC_FirstM87Results_Apr2019/uvfits/"
DESI_URL = "https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/230/23040/"
VM_URL = "sftp://lab-data/home/researcher/hallmark-exports/openuniverse2024/stars/"
DES_FILE = "2pt_NG_final_2ptunblind_02_24_21_wnz_redmagic_covupdate.fits"
DES_LOCAL = Path.home() / "datasets" / DES_FILE


This helper prints the plan and accepts only an explicit `y` before transferring files. An empty selection, a declined plan, or reported download failures raise an exception so the workflow stops. File sizes come from the catalog; a duration remains unknown without a supplied transfer rate.

In [ ]:
def download_after_review(repo, plan):
    print(plan.summary())
    if not plan.items:
        raise RuntimeError("No files selected; check the source and filter")
    if input("Download this selection? [y/N] ").strip().lower() != "y":
        raise RuntimeError("Download declined; catalog remains available")
    result = repo.download(plan, approved=True, progress=True)
    if result["failed"]:
        raise RuntimeError(result["errors"])
    return result

## 1. CyVerse: EHT M87 visibility provenance

Use one April 5 low-band UVFITS file from the [EHT M87 release](https://github.com/eventhorizontelescope/2019-D01-01). A scientific analysis could inspect positive-weight visibility amplitudes and compare the two frequency bands before imaging. This example records a QA choice without recalibrating visibilities or reconstructing an image.

Catalog that file by its URL in the release's `uvfits/` directory; the CyVerse backend is selected from the URL. Review the plan and approve its download into the new worktree.

In [ ]:
EHT_FILE = "SR1_M87_2017_095_lo_hops_netcal_StokesI.uvfits"
eht_dir = ROOT / "eht"
eht = Repo.init(eht_dir)
eht.add(EHT_URL + EHT_FILE)
eht.commit("Catalog EHT input")
download_after_review(eht, eht.plan_download(all_files=True))

Track the QA settings as a local template beside the cataloged input and commit them as a baseline, then record another choice on `eht-provenance`. Switching back restores the original settings; checkout leaves the downloaded input in place. The final checks confirm both the restoration and the unchanged input checksum.

In [ ]:
eht_settings = eht_dir / "analysis-settings.txt"
eht_settings.write_text("qa=positive-weights\n", encoding="utf-8")
eht.add("analysis-settings.txt")
eht.commit("Record EHT QA settings")
eht_base = eht.branches()["current"]
eht_hash = Repo.checksum(eht_dir / EHT_FILE)
eht.checkout("eht-provenance")
eht_settings.write_text("qa=positive-weights-and-amplitude-review\n", encoding="utf-8")
eht.add(".")
eht.commit("Record amplitude-review choice")
eht.checkout(eht_base)
assert eht_settings.read_text(encoding="utf-8") == "qa=positive-weights\n"
assert Repo.checksum(eht_dir / EHT_FILE) == eht_hash

A worktree lets you keep an alternative branch open alongside the baseline. Create its sibling directory, open it with `Repo(path)`, and commit another settings value there. The original worktree retains its baseline settings. The new worktree restores the tracked settings file; the remote input is cataloged there but not downloaded.

In [ ]:
eht.add_worktree("eht-alternate-qa")
eht_other = Repo(ROOT / "eht-alternate-qa")
other_settings = eht_other.worktree / "analysis-settings.txt"
other_settings.write_text("qa=band-consistency-review\n", encoding="utf-8")
eht_other.add(".")
eht_other.commit("Record alternative QA settings in a worktree")
assert eht_settings.read_text(encoding="utf-8") == "qa=positive-weights\n"

## 2. DESI: spectrum inspection and redshift-quality choices

Select only `redrock-main-dark-23040.fits` from [DR1 Iron HEALPixel 23040](https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/230/23040/). A scientific analysis could compare redshift-quality thresholds after checking the redshift and fibermap records by TARGETID. Inspecting spectra would also require the corresponding coadd, which this example does not download.

The settings below illustrate a threshold comparison, not official DESI LSS selection cuts. See the [redrock data model](https://desidatamodel.readthedocs.io/en/latest/DESI_SPECTRO_REDUX/SPECPROD/healpix/SURVEY/PROGRAM/PIXGROUP/PIXNUM/redrock-SURVEY-PROGRAM-PIXNUM.html) and [DR1 known issues](https://data.desi.lbl.gov/doc/releases/dr1/known-issues/) for scientific use.

Catalog the redrock file by its HTTPS URL, review the plan, and approve its download.

In [ ]:
DESI_FILE = "redrock-main-dark-23040.fits"
desi_dir = ROOT / "desi"
desi = Repo.init(desi_dir)
desi.add(DESI_URL + DESI_FILE)
desi.commit("Catalog DESI input")
download_after_review(desi, desi.plan_download(all_files=True))

Commit an illustrative DELTACHI2 threshold beside the cataloged input. Change only that threshold on `desi-quality`, commit it, then restore the baseline and check the unchanged input. No galaxy selection or FITS rewrite occurs here.

In [ ]:
desi_settings = desi_dir / "analysis-settings.txt"
desi_settings.write_text("min_deltachi2=25\n", encoding="utf-8")
desi.add("analysis-settings.txt")
desi.commit("Record DESI quality settings")
desi_base = desi.branches()["current"]
desi_hash = Repo.checksum(desi_dir / DESI_FILE)
desi.checkout("desi-quality")
desi_settings.write_text("min_deltachi2=40\n", encoding="utf-8")
desi.add(".")
desi.commit("Record stricter redshift-quality setting")
desi.checkout(desi_base)
assert desi_settings.read_text(encoding="utf-8") == "min_deltachi2=25\n"
assert Repo.checksum(desi_dir / DESI_FILE) == desi_hash

## 3. SSH/SFTP: Roman–Rubin products

Use the single `pointsource_10307.parquet` input from an existing export of the [OpenUniverse2024 shared-sky catalogs](https://irsa.ipac.caltech.edu/data/theory/openuniverse2024/roman/preview/roman_rubin_cats_v1.1.2_faint/). A scientific analysis could join source IDs with the separate flux table and compare Rubin color or magnitude selections. That additional table and the scientific processing are outside this example.

The [CLI guide](scientific_workflows_cli.md#3-sshsftp-romanrubin-products-over-sftp) shows an illustrative `lab-data` SSH configuration. Replace its host, user, and key with your own settings and verify the server's host key. Set `VM_URL` to an export you already have permission to read; the notebook does not provision a VM or prepare its data.

Catalog the selected file by its URL in the export's `stars/` directory, then review and approve its SFTP download. Other workflows do not use this SSH connection.

In [ ]:
VM_FILE = "pointsource_10307.parquet"
vm_dir = ROOT / "roman-rubin"
vm = Repo.init(vm_dir)
vm.add(VM_URL + VM_FILE)
vm.commit("Catalog Roman-Rubin input")
download_after_review(vm, vm.plan_download(all_files=True))

Keep the simulation input unchanged and record an intended magnitude limit in the settings file. Commit a different limit on `roman-rubin-selection`, then return to the baseline and verify both the settings and input checksum. No simulated-source selection is applied.

In [ ]:
vm_settings = vm_dir / "analysis-settings.txt"
vm_settings.write_text("magnitude_limit=24\n", encoding="utf-8")
vm.add("analysis-settings.txt")
vm.commit("Record Roman-Rubin selection settings")
vm_base = vm.branches()["current"]
vm_hash = Repo.checksum(vm_dir / VM_FILE)
vm.checkout("roman-rubin-selection")
vm_settings.write_text("magnitude_limit=23\n", encoding="utf-8")
vm.add(".")
vm.commit("Record brighter magnitude-limit setting")
vm.checkout(vm_base)
assert vm_settings.read_text(encoding="utf-8") == "magnitude_limit=24\n"
assert Repo.checksum(vm_dir / VM_FILE) == vm_hash

## +1. Local init: DES Y3 covariance-aware analysis versions

Start with your existing local `2pt_NG_final_2ptunblind_02_24_21_wnz_redmagic_covupdate.fits` file. This workflow makes no server connection. See the [DES Y3 papers](https://www.darkenergysurvey.org/des-year-3-cosmology-results-papers/) for the published analysis.

A scientific angular selection must apply the same retained rows to the data vector and both covariance axes, retaining the corresponding redshift-distribution information. The thresholds below only illustrate versioned settings; they are not the DES team's bin-dependent scale cuts and do not create a reduced likelihood.

Set `DES_LOCAL` to your input. Copy it into a new experiment directory and initialize a local repository there, leaving the original file untouched.

In [ ]:
assert DES_LOCAL.is_file(), "Set DES_LOCAL to your existing DES input"
des_dir = ROOT / "des"
des_dir.mkdir()
des_file = Path(shutil.copy2(DES_LOCAL, des_dir / DES_LOCAL.name))
des = Repo.init(des_dir)

Commit the copied input and angular settings. Record a second threshold on `des-angular-cut`, then restore the baseline and verify that both input copies still have the original checksum. This changes the text settings only; no covariance or data-vector processing occurs.

In [ ]:
des_settings = des_dir / "analysis-settings.txt"
des_settings.write_text("theta_min_arcmin=2.5\n", encoding="utf-8")
des.add("{name}")
des.commit("Preserve DES input and angular settings")
des_base = des.branches()["current"]
des_hash = Repo.checksum(DES_LOCAL)
des.checkout("des-angular-cut")
des_settings.write_text("theta_min_arcmin=5\n", encoding="utf-8")
des.add(".")
des.commit("Record alternative angular threshold")
des.checkout(des_base)
assert des_settings.read_text(encoding="utf-8") == "theta_min_arcmin=2.5\n"
assert Repo.checksum(des_file) == Repo.checksum(DES_LOCAL) == des_hash

## Inspect disk use and keep or remove your experiment

The plan reports available transfer sizes before approval. Allow room for each downloaded input; remote inputs are cataloged by URL and checksum rather than stored again in `.hm/objects`. The DES workflow copies your existing input once and stores a version in `.hm/objects`. Changing these small settings files does not make new versions of the unchanged scientific inputs.

Keep `ROOT` to retain the experiments and their history. To discard them, remove only the new experiment directories you created, including `eht-alternate-qa` if you ran that cell. Do not delete a shared dataset directory or remove only a worktree's `.hm` link.

## Validation record

For this revision, all 11 code cells were run in order on Python 3.13 against tiny local HTTP fixtures that mirror the EHT, DESI and export paths, with the SFTP export's URL served over HTTP and each approval prompt answered `y`. The checks covered cataloging, published checksums, approval, settings commits, worktree isolation, input checksums, and branch recovery. SFTP cataloging and transfer are covered by the OpenSSH integration tests with a loopback server.

The source notebook passes nbformat validation, keeps its outputs cleared, and requires explicit download approval. The matching [CLI guide](scientific_workflows_cli.md#disk-inspection-cleanup-and-validation) records the CLI and documentation checks. These fixtures do not validate scientific processing, public-server availability, or public download sizes; no research servers were contacted and no public datasets were downloaded.